In [1]:
!pip install -q transformers datasets accelerate bitsandbytes peft

In [2]:
with open("dataset.txt", "r") as f:
    raw_text = f.read()

examples = [ex.strip() for ex in raw_text.split("\n\n") if ex.strip()]

print(f"Successfully loaded {len(examples)} examples.")

Successfully loaded 240 examples.


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # Standard for Llama models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("Model ready for LoRA training!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model ready for LoRA training!


In [4]:
def tokenize(example):
    outputs = tokenizer(
        example,
        max_length=2048,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = outputs["input_ids"].squeeze(0)
    attention_mask = outputs["attention_mask"].squeeze(0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()
    }

tokenized_examples = [tokenize(ex) for ex in examples]

print(f"First example shape: {tokenized_examples[0]['input_ids'].shape}")

First example shape: torch.Size([2048])


In [5]:
from datasets import Dataset

list_for_dataset = []
for ex in tokenized_examples:
    list_for_dataset.append({
        "input_ids": ex["input_ids"].tolist(),
        "attention_mask": ex["attention_mask"].tolist(),
        "labels": ex["labels"].tolist()
    })

dataset = Dataset.from_list(list_for_dataset)
dataset.set_format("torch")

print(dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 240
})


In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    prediction_loss_only=True,
    report_to="none"
)

In [7]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,8.264388
20,8.272437
30,6.204544
40,5.506802
50,5.079509
60,4.540107
70,4.067160
80,3.322475
90,3.451032


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=90, training_loss=5.412050374348959, metrics={'train_runtime': 1398.2298, 'train_samples_per_second': 0.515, 'train_steps_per_second': 0.064, 'total_flos': 9162669152010240.0, 'train_loss': 5.412050374348959, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
save_path = "./minecraft-model-forge-finetuned"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

!zip -r minecraft-model-forge-finetuned.zip {save_path}

print("Training finished! Download the .zip file from the sidebar.")

  adding: minecraft-model-forge-finetuned/ (stored 0%)
  adding: minecraft-model-forge-finetuned/README.md (deflated 66%)
  adding: minecraft-model-forge-finetuned/tokenizer.json (deflated 85%)
  adding: minecraft-model-forge-finetuned/adapter_model.safetensors (deflated 8%)
  adding: minecraft-model-forge-finetuned/training_args.bin (deflated 53%)
  adding: minecraft-model-forge-finetuned/adapter_config.json (deflated 57%)
  adding: minecraft-model-forge-finetuned/tokenizer_config.json (deflated 46%)
Training finished! Download the .zip file from the sidebar.
